In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 비저항 예측 인공지능 학습 모델

### 1. 라이브러리 불러오기

In [3]:
!pip install keras


  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 10.5 MB/s eta 0:00:00
Using cached absl_py-2.3.1-py3-none-any.whl (135 kB)
Using cached namex-0.1.0-py3-none-any.whl (5.9 kB)

   ---------------- ----------------------- 2/5 [ml-dtypes]
   -------------------------------- ------- 4/5 [keras]
   -------------------------------- ------- 4/5 [keras]
   -------------------------------- ------- 4/5 [keras]
   -------------------------------- ------- 4/5 [keras]
   -------------------------------- ------- 4/5 [keras]
   -------------------------------- ------- 4/5 [keras]
   -------------------------------- ------- 4/5 [keras]
   -------------------------------- ------- 4/5 [keras]
   -------------------------------- ------- 4/5 [keras]
   -------------------------------- ------- 4/5 [keras]



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install tensorflow


  Using cached tensorflow-2.19.0-cp310-cp310-win_amd64.whl.metadata (4.1 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-5.29.5-cp310-abi3-win_amd64.whl.metadata (592 bytes)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached grpcio-1.74.0-cp310-cp310-win_amd64.whl.metadata (4.0 kB)
  Using cached tensorboard-2.19.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached h5py-3.14.0-cp310-cp310-win_amd64.whl.metadata (2.7 kB)
  Using cached tensorflow_io_gcs_filesystem-0.31.0-cp310-cp310-win_amd64.whl.metadata (14 kB)
  Using cached tensorboard_data_server-0.7.2-py3-


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import keras
from keras.models import Sequential
from keras.optimizers import Adam, Nadam, SGD, Adamax, Adagrad
from keras.layers import Dense
from keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf

from time import time, ctime
ctime(time())

ImportError: Traceback (most recent call last):
  File "C:\Users\Admin\anaconda3\lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: DLL 초기화 루틴을 실행할 수 없습니다.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

### 2. 학습용 데이터 SET 불러오기

In [2]:
# Importing the dataset
df_r = pd.read_csv(r'Resistivity_data_set.csv') # Error 발생할 경우 파일 경로 확인 필수
df_r.head()

,Number,X,Y,Al,Ti,Cr,Fe,Co,Ni,Cu,...,Thickness,Resistivity,Ex_resistivity,ravg,delta,dHmix,ENavg,dEN,N,Compo
0,1,-41,-11,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.443489,8.166252,1.926252,0.125,0.0,0.0,1.88,0.0,1,Co
1,2,-41,-9,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.357119,8.159013,1.919013,0.125,0.0,0.0,1.88,0.0,1,Co
2,3,-41,-7,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.270749,8.107711,1.867711,0.125,0.0,0.0,1.88,0.0,1,Co
3,4,-41,-5,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.184379,8.100512,1.860512,0.125,0.0,0.0,1.88,0.0,1,Co
4,5,-41,-3,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.098008,8.093313,1.853313,0.125,0.0,0.0,1.88,0.0,1,Co


### 3. 학습용 데이터 전처리

In [3]:
# 학습용 데이터의 크기를 확인한다.
print('학습용 데이터의 크기: ', df_r.shape)

학습용 데이터의 크기:  (65484, 27)


In [4]:
# 학습용 데이터의 특성값을 확인한다.

print('데이터 SET의 전체 특성의 개수: ', df_r.columns.nunique())
print('__________________________________________________________________________')
print(df_r.columns.unique())

데이터 SET의 전체 특성의 개수:  27
__________________________________________________________________________
Index(['Number', 'X', 'Y', 'Al', 'Ti', 'Cr', 'Fe', 'Co', 'Ni', 'Cu', 'Zr',
       'Mo', 'W', 'Mn', 'Si', 'Mg', 'Resistance', 'Thickness', 'Resistivity',
       'Ex_resistivity', 'ravg', 'delta', 'dHmix', 'ENavg', 'dEN', 'N',
       'Compo'],
      dtype='object')


In [5]:
# 각 컬럼(특성)의 정보를 확인한다.
df_r.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65484 entries, 0 to 65483
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Number          65484 non-null  int64  
 1   X               65484 non-null  int64  
 2   Y               65484 non-null  int64  
 3   Al              65484 non-null  float64
 4   Ti              65484 non-null  float64
 5   Cr              65484 non-null  float64
 6   Fe              65484 non-null  float64
 7   Co              65484 non-null  float64
 8   Ni              65484 non-null  float64
 9   Cu              65484 non-null  float64
 10  Zr              65484 non-null  float64
 11  Mo              65484 non-null  float64
 12  W               65484 non-null  float64
 13  Mn              65484 non-null  int64  
 14  Si              65484 non-null  int64  
 15  Mg              65484 non-null  float64
 16  Resistance      65484 non-null  float64
 17  Thickness       65484 non-null 

In [6]:
# 각 특성별 데이터의 통계값을 확인한다.
df_r.describe().T

,count,mean,std,min,25%,50%,75%,max
Number,65484.0,32742.500000,18903.746851,1.000000,16371.750000,32742.500000,49113.250000,65484.000000
X,65484.0,0.000000,21.371960,-41.000000,-17.000000,0.000000,17.000000,41.000000
Y,65484.0,0.000000,21.238295,-41.000000,-17.000000,0.000000,17.000000,41.000000
Al,65484.0,8.878069,16.176507,0.000000,0.000000,0.000000,12.557294,76.168290
Ti,65484.0,11.218753,19.143389,0.000000,0.000000,0.000000,18.423244,96.910260
Cr,65484.0,14.010004,22.319159,-0.803022,0.000000,0.000000,24.330750,86.389780
Fe,65484.0,8.590410,19.383900,0.000000,0.000000,0.000000,0.000000,97.245963
Co,65484.0,19.856873,35.817582,0.000000,0.000000,0.000000,21.462232,100.000000
Ni,65484.0,9.246283,18.553823,0.000000,0.000000,0.000000,11.955932,94.301490
Cu,65484.0,13.995509,26.183223,0.000000,0.000000,0.000000,16.256139,97.661327


In [7]:
# 인공지능 학습용 데이터셋에 조성 조합을 확인한다.
df_r['Compo'].unique()

array(['Co', 'Co/Cu', 'Co/Ni', 'Zr/Cu/Al', 'Zr/Cu/Ti', 'Ni/Mo/W',
       'Co/Cr/Ti', 'Ni/Fe/Cr', 'Cu/Zr/Ti', 'Al/Cu/Zr', 'Zr/Cu/Ni/Al',
       'Al/Ti/Cr', 'Mg/Al/Zr', 'Mg/Al/Cr', 'Mg/Al/Fe', 'Mg/Al/Ti',
       'Mg/Ti/Cr', 'Ti/Cr/Fe', 'Ti/Cr/Ni', 'Cu/Zr', 'Cu/Ti', 'Ni/Zr',
       'Ni/Ti', 'Al/Ti/Fe', 'Al/Cr/Co', 'Al/Co/Mo', 'Cr/Ti/Co/Mo',
       'Cr/Ti/Co', 'Mg/Ti', 'Mg/Cu', 'Mg/Cu/Ti'], dtype=object)

In [8]:
# 인공지능 학습을 위하여 각 컬럼별 Null 값을 확인한다.
df_r.isnull().any()

Number            False
X                 False
Y                 False
Al                False
Ti                False
Cr                False
Fe                False
Co                False
Ni                False
Cu                False
Zr                False
Mo                False
W                 False
Mn                False
Si                False
Mg                False
Resistance        False
Thickness         False
Resistivity       False
Ex_resistivity    False
ravg              False
delta             False
dHmix             False
ENavg             False
dEN               False
N                 False
Compo             False
dtype: bool

### 4. 인공지능 학습용 모델

#### 4-1. 피쳐값 지정

전체 27개 피쳐값 가운데 Number, N, Compo 값은 인공지능 학습에 크게 중요하지 않으므로 사용하지 않는다.
여러 피쳐값을 조합하여 각 조성에 따른 Ex_resistivity를 예측한다.

##### Case 1. Number, N, Compo를 제외한 22개의 피쳐값을 사용하여 Ex_resistivity 예측
##### Case 2. Resistance, Thickness, Resistivity, ravg, delta, dHmix, ENavg, dEN 사용하여 Ex_resistivity 예측
##### Case 3. Resistivity, delta, dHmix, ENavg, dEN 사용하여 Ex_resistivity 예측
##### Case 4. Resistivity, delta, ENavg 사용하여 Ex_resistivity 예측
#####  Case 5. delta 사용하여 Ex_resistivity 예측

** 참고사항, ρ(로우)는 물리학에서 비저항(Resistivity)을 나타낸다. **

In [9]:
# 사용 안하는 3가지 피쳐값을 제외한다.
df = df_r.drop(['Number', 'N', 'Compo'], axis = 1)
df.head()

,X,Y,Al,Ti,Cr,Fe,Co,Ni,Cu,Zr,...,Mg,Resistance,Thickness,Resistivity,Ex_resistivity,ravg,delta,dHmix,ENavg,dEN
0,-41,-11,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,...,0.0,0.185,97.443489,8.166252,1.926252,0.125,0.0,0.0,1.88,0.0
1,-41,-9,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,...,0.0,0.185,97.357119,8.159013,1.919013,0.125,0.0,0.0,1.88,0.0
2,-41,-7,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,...,0.0,0.184,97.270749,8.107711,1.867711,0.125,0.0,0.0,1.88,0.0
3,-41,-5,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,...,0.0,0.184,97.184379,8.100512,1.860512,0.125,0.0,0.0,1.88,0.0
4,-41,-3,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,...,0.0,0.184,97.098008,8.093313,1.853313,0.125,0.0,0.0,1.88,0.0


In [10]:
# AI 성능 평가를 위한 함수 

def score_matrix(y_real, y_pred, X_test):
    print("MAE: ", mean_absolute_error(y_real, y_pred))
    SSE = np.sum((y_real - y_pred)**2)
    SSR = np.sum((y_pred - np.mean(y_real))**2)
    print("R2 Score :", 1-SSE/SSR)
    pred_rsq = 1 - np.sum(np.square(y_real-y_pred)) / np.var(y_real) / y_real.size
    print("Predicted R2 Score :", pred_rsq)
    p = X_test.shape[1]
    adj_r2 = 1-(SSE/SSR) * (len(y_real)-1) / (len(y_real) - p - 1)
    print("adjust R2 Score :", adj_r2)
    
data_set = []

In [11]:
# Case 1. Number, N, Compo를 제외한 22개의 피쳐값을 사용하여 Ex_resistivity 예측
X1 = np.array(df[[ 'X', 'Y', 'Al', 'Ti', 'Cr', 'Fe', 'Co', 'Ni', 'Cu', 'Zr',
       'Mo', 'W', 'Mn', 'Si', 'Mg', 'Resistance', 'Thickness',
       'ravg', 'delta', 'dHmix', 'ENavg', 'dEN']])
X1 = X1.astype(float)
y1 = np.array(df["Ex_resistivity"])
y1 = y1.astype(float)
data_set.append(train_test_split(X1, y1, test_size = 0.4, random_state=42))

In [12]:
# Case 2. Resistance, Thickness, ravg, delta, dHmix, ENavg, dEN 사용하여 Ex_resistivity 예측
X2 = np.array(df[["Thickness", "ravg", "delta", "dHmix", "ENavg", "dEN"]]) # 6 Features
X2 = X2.astype(float)
y2 = np.array(df["Ex_resistivity"])
y2 = y2.astype(float)
data_set.append(train_test_split(X2, y2, test_size = 0.2, random_state=42))

In [13]:
# Case 3. "delta", "dHmix", "ENavg", "dEN" 4가지 특성값을 사용하여 Ex_resistivity 예측
X3 = np.array(df[["delta", "dHmix", "ENavg", "dEN"]]) # 4 Features
X3 = X3.astype(float)
y3 = np.array(df["Ex_resistivity"])
y3 = y3.astype(float)
data_set.append(train_test_split(X3, y3, test_size = 0.2, random_state=38))

In [14]:
# Case 4. delta, ENavg 사용하여 Ex_resistivity 예측
X4 = np.array(df[[ "delta", "ENavg"]]) # 2 Features
X4 = X4.astype(float)
y4 = np.array(df["Ex_resistivity"])
y4 = y4.astype(float)
data_set.append(train_test_split(X4, y4, test_size = 0.2, random_state=38))

In [15]:
# Case 5. delta 사용하여 Ex_resistivity 예측
X5 = np.array(df[["delta"]]) # 1 Features
X5 = X5.astype(float)
y5 = np.array(df["Ex_resistivity"])
y5 = y5.astype(float)
data_set.append(train_test_split(X5, y5, test_size = 0.2, random_state=38))

In [16]:
# Model 1. LinearRegression 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model = LinearRegression()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1
MAE:  23.452422820665664
R2 Score : 0.7978049918242142
Predicted R2 Score : 0.8331974253935922
adjust R2 Score : 0.7976350216213229
Features Case_ 2
MAE:  50.0211201962239
R2 Score : -0.19865519850243296
Predicted R2 Score : 0.4567039982208978
adjust R2 Score : -0.1992046202893707
Features Case_ 3
MAE:  50.281681290510505
R2 Score : -0.1975510523910171
Predicted R2 Score : 0.45905166436666534
adjust R2 Score : -0.19791694027747941
Features Case_ 4
MAE:  55.1036843457772
R2 Score : -0.6592063255840277
Predicted R2 Score : 0.3843259417368726
adjust R2 Score : -0.6594597556016821
Features Case_ 5
MAE:  65.16122693398016
R2 Score : -8.969799354593933
Predicted R2 Score : 0.09582344723630387
adjust R2 Score : -8.970560698569084


In [17]:
# Model 2. DecisionTreeRegressor 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model = DecisionTreeRegressor()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1
MAE:  1.56098487552848
R2 Score : 0.9979848154203198
Predicted R2 Score : 0.9979929757336039
adjust R2 Score : 0.9979831214055419
Features Case_ 2
MAE:  12.271758164606716
R2 Score : 0.8805808293828196
Predicted R2 Score : 0.8829598526056329
adjust R2 Score : 0.8805260917950654
Features Case_ 3
MAE:  17.479575868279834
R2 Score : 0.8293201244922749
Predicted R2 Score : 0.8310602858712223
adjust R2 Score : 0.8292679766537452
Features Case_ 4
MAE:  25.33971787231464
R2 Score : 0.7178067051263444
Predicted R2 Score : 0.7172871472158089
adjust R2 Score : 0.7177636024388732
Features Case_ 5
MAE:  47.57978222984883
R2 Score : 0.32416163025361355
Predicted R2 Score : 0.33863092624421987
adjust R2 Score : 0.3241100198397344


In [18]:
# Model 3. RandomForestRegressor 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model =  RandomForestRegressor(n_estimators=25, n_jobs=-1, verbose=1)
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    1.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.


MAE:  0.9260813664402943
R2 Score : 0.9986813724887389
Predicted R2 Score : 0.9986897711368444
adjust R2 Score : 0.9986802640173298
Features Case_ 2


[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    1.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.


MAE:  10.227349680153543
R2 Score : 0.9140119899070421
Predicted R2 Score : 0.9174887687875789
adjust R2 Score : 0.9139725759986725
Features Case_ 3


[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    1.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.


MAE:  15.322197380902544
R2 Score : 0.8650069338484058
Predicted R2 Score : 0.8716248111910581
adjust R2 Score : 0.8649656894041187
Features Case_ 4


[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    0.7s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.


MAE:  22.2333717695139
R2 Score : 0.7824486022882745
Predicted R2 Score : 0.8042689151128029
adjust R2 Score : 0.7824153731149567
Features Case_ 5
MAE:  43.374289134022725
R2 Score : 0.33214297905027657
Predicted R2 Score : 0.46629031510862007
adjust R2 Score : 0.3320919781322965


[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    0.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished


In [19]:
# Model 4. Lasso 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model =  Lasso()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1
MAE:  24.86654994030944
R2 Score : 0.75091295799235
Predicted R2 Score : 0.8083379107620134
adjust R2 Score : 0.7507035691679196
Features Case_ 2
MAE:  54.28835291741273
R2 Score : -0.8297887898492153
Predicted R2 Score : 0.3970614335781516
adjust R2 Score : -0.8306275012884128
Features Case_ 3
MAE:  53.87709042252379
R2 Score : -0.7884161432888352
Predicted R2 Score : 0.410770535425361
adjust R2 Score : -0.7889625582424828
Features Case_ 4
MAE:  55.71392316573303
R2 Score : -1.296165017193759
Predicted R2 Score : 0.3744147507466924
adjust R2 Score : -1.2965157373735656
Features Case_ 5
MAE:  65.33572885504532
R2 Score : -14.108211548106496
Predicted R2 Score : 0.09173024368950788
adjust R2 Score : -14.109365287056333


In [20]:
# Model 5. Lasso 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model =  Ridge()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1
MAE:  23.419840897300453
R2 Score : 0.7973280030669234
Predicted R2 Score : 0.8330538135132352
adjust R2 Score : 0.7971576318953011
Features Case_ 2
MAE:  50.04426647444901
R2 Score : -0.20229748316203633
Predicted R2 Score : 0.45673013409614405
adjust R2 Score : -0.2028485744453803
Features Case_ 3
MAE:  50.294975232647815
R2 Score : -0.20020985281207637
Predicted R2 Score : 0.45904347524175715
adjust R2 Score : -0.2005765530420831
Features Case_ 4
MAE:  55.108325285447066
R2 Score : -0.6629224033917749
Predicted R2 Score : 0.3843116009361778
adjust R2 Score : -0.6631764010095222
Features Case_ 5
MAE:  65.16122085464991
R2 Score : -8.978652659494806
Predicted R2 Score : 0.09582114336349024
adjust R2 Score : -8.979414679552804


In [21]:
ctime(time())

'Fri Dec 16 09:23:11 2022'